# Day 44 — NumPy vectorisation and automatic differentiation: fast array code, and gradients you can trust

Summing the squares of a million floats takes tens of milliseconds in a Python loop and a
fraction of a millisecond with `np.dot` - the same arithmetic, with the loop moved into C
over one buffer of raw doubles. The first half of this notebook is about why, and about the
NumPy mistakes that give wrong numbers without an error: broadcasting a `(n, 1)` against an
`(n,)`, `uint8` wrap-around, and writes into a copy.

The second half builds gradients three ways - difference quotients, dual numbers (forward
mode) and a small reverse-mode autograd - and shows the two autograd bugs that still return
a number: overwriting instead of adding gradients, and walking the graph without a
topological order (Day 13).

In [1]:
import math
import random
import time

import numpy as np

## 1. One loop in Python, one loop in C

A Python list holds pointers to separate float objects; a NumPy array holds one buffer of
raw doubles and a single type, so one C loop can run over it.

In [2]:
def best_of(fn, repeat=5):
    best = float('inf')
    for _ in range(repeat):
        t = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t)
    return best


# ---------------------------------------------------------------- 1. loops vs vectorised
def sum_squares_loop(xs):
    s = 0.0
    for x in xs:
        s += x * x
    return s


def sum_squares_numpy(a):
    return float(np.dot(a, a))


a = np.random.default_rng(44).random(1_000_000)
xs = a.tolist()
t_loop = best_of(lambda: sum_squares_loop(xs), 3)
t_mul = best_of(lambda: (a * a).sum())
t_dot = best_of(lambda: sum_squares_numpy(a))
print(f'loop {t_loop*1000:.1f} ms, (a*a).sum() {t_mul*1000:.2f} ms, np.dot {t_dot*1000:.2f} ms')
print('same answer:', round(sum_squares_loop(xs), 6), round(sum_squares_numpy(a), 6))

loop 52.8 ms, (a*a).sum() 0.96 ms, np.dot 0.10 ms
same answer: 333637.313928 333637.313928


## 2. Shape and strides

An array is a buffer plus a shape plus *strides* - how many bytes to jump per step along
each axis. Transposing and slicing only change the strides: the result is a view of the
same memory. Reshaping a transposed array cannot be expressed with strides, so it copies.

In [3]:
def layout(a, base=None):
    """What NumPy actually stores: one buffer, a shape and the byte step per axis."""
    return {'shape': a.shape, 'strides': a.strides,
            'shares memory': bool(base is not None and np.shares_memory(a, base)),
            'C-contiguous': bool(a.flags['C_CONTIGUOUS'])}


m = np.arange(12).reshape(3, 4)
for name, v in (('m', m), ('m.T', m.T), ('m[:, ::2]', m[:, ::2]), ('m.T.reshape(12)', m.T.reshape(12))):
    print(f'{name:<16}', layout(v, m))

m                {'shape': (3, 4), 'strides': (32, 8), 'shares memory': True, 'C-contiguous': True}
m.T              {'shape': (4, 3), 'strides': (8, 32), 'shares memory': True, 'C-contiguous': False}
m[:, ::2]        {'shape': (3, 2), 'strides': (32, 16), 'shares memory': True, 'C-contiguous': False}
m.T.reshape(12)  {'shape': (12,), 'strides': (8,), 'shares memory': False, 'C-contiguous': True}


## 3. Broadcasting, and the shape bug it hides

Shapes are compared from the right; each pair must be equal or contain a 1, which is
stretched. A model that returns predictions of shape `(n, 1)` subtracted from targets of
shape `(n,)` silently produces an `(n, n)` table.

In [4]:
def broadcast_shape(s1, s2):
    """NumPy's rule, written out: align from the right; each pair must be equal or 1."""
    out = []
    for i in range(1, max(len(s1), len(s2)) + 1):
        a = s1[-i] if i <= len(s1) else 1
        b = s2[-i] if i <= len(s2) else 1
        if a != b and 1 not in (a, b):
            raise ValueError(f'shapes {s1} and {s2} do not broadcast')
        out.append(max(a, b))
    return tuple(reversed(out))


def mse(pred, y):
    return float(((pred - y) ** 2).mean())


print(broadcast_shape((3,), (3, 1)), broadcast_shape((5, 1, 4), (3, 1)))
y = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
pred = (y + 0.1).reshape(-1, 1)
print('pred - y shape:', (pred - y).shape)
print('MSE right:', mse(pred.ravel(), y), '  MSE with (5, 1):', mse(pred, y))

(3, 3) (5, 3, 4)
pred - y shape: (5, 5)
MSE right: 0.009999999999999981   MSE with (5, 1): 4.01


## 4. Wrap-around and copies

`uint8` arithmetic wraps at 256, and fancy indexing (`v[[0, 2]]`) returns a copy, so
writing into it changes nothing.

In [5]:
a8 = np.array([200, 100, 250], dtype=np.uint8)
b8 = np.array([100, 100, 10], dtype=np.uint8)
print('uint8 sum:', (a8 + b8).tolist(), ' average:', ((a8 + b8) // 2).tolist(),
      ' in int16:', ((a8.astype(np.int16) + b8) // 2).tolist())
v = np.arange(6); s = v[::2]; s[:] = 0; print('slice is a view:', v.tolist())
v = np.arange(6); v[[0, 2]][0] = 99; print('fancy index is a copy:', v.tolist())

uint8 sum: [44, 200, 4]  average: [22, 100, 2]  in int16: [150, 100, 130]
slice is a view: [0, 1, 0, 3, 0, 5]
fancy index is a copy: [0, 1, 2, 3, 4, 5]


## 5. Pairwise distances

Broadcasting turns the double loop into one expression but builds an `(n, n, d)` array;
the identity `|a-b|^2 = |a|^2 + |b|^2 - 2a.b` needs one matrix product - and a clip,
because rounding can make a zero distance slightly negative.

In [6]:
def pairwise_loop(x):
    n = len(x)
    out = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            out[i][j] = math.sqrt(sum((x[i][k] - x[j][k]) ** 2 for k in range(len(x[i]))))
    return out


def pairwise_broadcast(x):
    diff = x[:, None, :] - x[None, :, :]          # (n, n, d) - every pair, materialised
    return np.sqrt((diff ** 2).sum(axis=-1))


def pairwise_gram(x, clip=True):
    """|a - b|^2 = |a|^2 + |b|^2 - 2 a.b : one matrix product, no (n, n, d) array."""
    sq = (x * x).sum(axis=1)
    d2 = sq[:, None] + sq[None, :] - 2.0 * (x @ x.T)
    if clip:
        d2 = np.maximum(d2, 0.0)                  # rounding can push a 0 to -2e-16
    with np.errstate(invalid='ignore'):
        return np.sqrt(d2)


pts = np.random.default_rng(7).random((300, 3))
print('loop vs broadcast max diff:', np.abs(pairwise_broadcast(pts) - np.array(pairwise_loop(pts.tolist()))).max())
many = np.random.default_rng(44).random((500, 3))
sq = (many * many).sum(1)
raw = sq[:, None] + sq[None, :] - 2 * many @ many.T
print('negative squared distances without clipping:', int((raw < 0).sum()))

loop vs broadcast max diff: 2.220446049250313e-16
negative squared distances without clipping: 63


## 6. Derivatives: difference quotients and dual numbers

The central difference `(f(x+h) - f(x-h)) / 2h` is accurate to about `h^2` - until `h` is
so small that the two values of `f` agree in almost every digit. Dual numbers carry the
derivative along with the value and are exact, but give one input's derivative per pass.

In [7]:
# ---------------------------------------------------------------- 2. derivatives by hand
def central_diff(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


def forward_diff(f, x, h):
    return (f(x + h) - f(x)) / h


class Dual:
    """Forward mode: carry (value, derivative) through every operation."""

    def __init__(self, val, der=0.0):
        self.val, self.der = val, der

    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.val + o.val, self.der + o.der)
    __radd__ = __add__

    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.val * o.val, self.der * o.val + self.val * o.der)
    __rmul__ = __mul__

    def sin(self):
        return Dual(math.sin(self.val), math.cos(self.val) * self.der)


for k in (1, 3, 5, 6, 8, 10, 12):
    h = 10.0 ** -k
    print(f'h = 1e-{k:<2} forward {abs(forward_diff(math.sin, 1.0, h) - math.cos(1)):.1e}'
          f'  central {abs(central_diff(math.sin, 1.0, h) - math.cos(1)):.1e}')
x = Dual(2.0, 1.0)
fx = x * x.sin() + 3 * x
print('dual:', fx.der, ' exact:', math.sin(2) + 2 * math.cos(2) + 3)

h = 1e-1  forward 4.3e-02  central 9.0e-04
h = 1e-3  forward 4.2e-04  central 9.0e-08
h = 1e-5  forward 4.2e-06  central 1.1e-11
h = 1e-6  forward 4.2e-07  central 2.8e-11
h = 1e-8  forward 3.0e-09  central 2.6e-09
h = 1e-10 forward 5.8e-08  central 5.8e-08
h = 1e-12 forward 4.3e-05  central 1.2e-05
dual: 3.077003753731397  exact: 3.077003753731397


## 7. A mini autograd (reverse mode)

Every `Value` remembers its parents and a small function that sends its gradient back to
them. `backward()` orders the graph topologically (Day 13) and runs those functions from
the output to the leaves, so every node has heard from all of its users before it passes
its own gradient on. Two bugs to watch: `grad = g` instead of `grad += g`, and recursing
into children without a topological order.

In [8]:
# ---------------------------------------------------------------- 3. a mini autograd
class Value:
    """A scalar that remembers how it was made, so it can send gradients back."""

    accumulate = True          # the right rule: a node used twice gets both contributions

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = tuple(_children)
        self._op = _op
        self.label = label
        self._backward = lambda: None

    def _wrap(self, o):
        return o if isinstance(o, Value) else type(self)(o)

    def _send(self, child, g):
        child.grad = child.grad + g if self.accumulate else g

    def __add__(self, o):
        o = self._wrap(o)
        out = type(self)(self.data + o.data, (self, o), '+')

        def _backward():
            self._send(self, out.grad)
            self._send(o, out.grad)
        out._backward = _backward
        return out

    def __mul__(self, o):
        o = self._wrap(o)
        out = type(self)(self.data * o.data, (self, o), '*')

        def _backward():
            self._send(self, o.data * out.grad)
            self._send(o, self.data * out.grad)
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float))
        out = type(self)(self.data ** k, (self,), f'**{k}')

        def _backward():
            self._send(self, k * self.data ** (k - 1) * out.grad)
        out._backward = _backward
        return out

    def exp(self):
        out = type(self)(math.exp(self.data), (self,), 'exp')

        def _backward():
            self._send(self, out.data * out.grad)
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = type(self)(t, (self,), 'tanh')

        def _backward():
            self._send(self, (1 - t * t) * out.grad)
        out._backward = _backward
        return out

    def relu(self):
        out = type(self)(max(0.0, self.data), (self,), 'relu')

        def _backward():
            self._send(self, (out.data > 0) * out.grad)
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, o):
        return self + (-self._wrap(o))

    def __truediv__(self, o):
        return self * self._wrap(o) ** -1

    def __radd__(self, o):
        return self + o

    def __rmul__(self, o):
        return self * o

    def __rsub__(self, o):
        return self._wrap(o) - self

    def __rtruediv__(self, o):
        return self._wrap(o) / self

    def __repr__(self):
        return f'Value({self.data:.4f}, grad={self.grad:.4f})'

    # -- the backward pass: reverse topological order (Day 13), every node once
    def topo(self):
        order, seen, stack = [], set(), [(self, False)]
        while stack:                       # iterative DFS: no recursion limit
            v, done = stack.pop()
            if done:
                order.append(v)
                continue
            if id(v) in seen:
                continue
            seen.add(id(v))
            stack.append((v, True))
            for c in v._prev:
                if id(c) not in seen:
                    stack.append((c, False))
        return order

    def backward(self):
        order = self.topo()
        for v in order:
            v.grad = 0.0
        self.grad = 1.0
        for v in reversed(order):
            v._backward()
        return order

    def backward_naive(self):
        """The tempting version: push my gradient to my children, then recurse into
        each child at once - before the child has heard from all of its parents."""
        self.grad = 1.0
        calls = [0]

        def go(v):
            calls[0] += 1
            v._backward()
            for c in v._prev:
                go(c)
        go(self)
        return calls[0]


class AssignValue(Value):
    """The same Value with `grad = g` instead of `grad += g`."""
    accumulate = False


def numeric_grads(f, xs, h=1e-6):
    out = []
    for i in range(len(xs)):
        up = list(xs)
        dn = list(xs)
        up[i] += h
        dn[i] -= h
        out.append((f(*up) - f(*dn)) / (2 * h))
    return out


def gradcheck(f, xs, h=1e-6):
    """Largest relative difference between autograd and central differences."""
    vs = [Value(x) for x in xs]
    f(*vs).backward()
    num = numeric_grads(lambda *a: f(*[Value(t) for t in a]).data, xs, h)
    worst = 0.0
    for v, n in zip(vs, num):
        worst = max(worst, abs(v.grad - n) / max(1e-12, abs(v.grad) + abs(n)))
    return worst


MICRO = dict(a=2.0, b=-3.0, c=10.0, f=-2.0)


def micrograd_example():
    a, b, c, f = (Value(MICRO[k], label=k) for k in 'abcf')
    e = a * b
    e.label = 'e'
    d = e + c
    d.label = 'd'
    L = d * f
    L.label = 'L'
    order = L.backward()
    return {v.label: (v.data, v.grad) for v in order}, [v.label for v in order]


res, order = micrograd_example()
print('order:', order)
print({k: v for k, v in res.items()})
f3 = lambda p, q, r: (p * q + r).tanh() * p + (q / r).exp() - p ** 2
print('gradcheck worst relative error:', gradcheck(f3, [0.5, -1.2, 2.0]))
x = Value(3.0); (x * x).backward()
xa = AssignValue(3.0); (xa * xa).backward()
print('d(x*x)/dx: +=', x.grad, '  =', xa.grad)
x = Value(3.0); q = x * 2; (q * q).backward(); print('(2x)^2 topological:', x.grad)
x = Value(3.0); q = x * 2; (q * q).backward_naive(); print('(2x)^2 naive recursion:', x.grad)

order: ['f', 'c', 'b', 'a', 'e', 'd', 'L']
{'f': (-2.0, 4.0), 'c': (10.0, -2.0), 'b': (-3.0, -4.0), 'a': (2.0, 6.0), 'e': (-6.0, -2.0), 'd': (4.0, -2.0), 'L': (-8.0, 1.0)}
gradcheck worst relative error: 1.310908327393625e-10
d(x*x)/dx: += 6.0   = 3.0
(2x)^2 topological: 24.0
(2x)^2 naive recursion: 48.0


## 8. Training: forgetting to zero the gradients

Gradients add up by design. Between optimisation steps they must be reset, or each step
uses the sum of every gradient so far.

In [9]:
# ---------------------------------------------------------------- 4. training: zero_grad
def make_line_data(n=20, seed=44):
    rng = random.Random(seed)
    xs = [rng.uniform(-1, 1) for _ in range(n)]
    ys = [3.0 * x - 0.5 + rng.gauss(0, 0.1) for x in xs]
    return xs, ys


def fit_line(xs, ys, steps=60, lr=0.1, zero_grad=True):
    """Gradient descent on w, b with Value; returns (w, b, loss history)."""
    w, b = Value(0.0, label='w'), Value(0.0, label='b')
    hist = []
    for _ in range(steps):
        loss = sum(((w * x + b) - y) ** 2 for x, y in zip(xs, ys)) * (1 / len(xs))
        if zero_grad:
            w.grad = b.grad = 0.0          # forget the previous step's gradient
        order = loss.topo()
        for v in order:
            if v is not w and v is not b:
                v.grad = 0.0
        loss.grad = 1.0
        for v in reversed(order):
            v._backward()
        w.data -= lr * w.grad
        b.data -= lr * b.grad
        hist.append(loss.data)
    return w.data, b.data, hist


xs, ys = make_line_data()
for zg in (True, False):
    w, b, hist = fit_line(xs, ys, zero_grad=zg)
    print(f'zero_grad={zg}: w={w:.3f} b={b:.3f} final loss {hist[-1]:.4f}')

zero_grad=True: w=3.020 b=-0.534 final loss 0.0083
zero_grad=False: w=6.089 b=-1.029 final loss 3.0924


## 9. The same gradient from a scalar graph and from one matrix formula

The Value graph for a 200 x 5 least-squares loss has thousands of nodes; the vectorised
gradient is `2/n X^T (Xw - y)`. Same numbers, very different cost - which is why real
autograd works on whole arrays.

In [10]:
# ---------------------------------------------------------------- 5. scalar graph vs matrix formula
def linreg_grad_numpy(X, y, w):
    r = X @ w - y
    return 2.0 / len(y) * (X.T @ r)


def linreg_grad_values(X, y, w):
    ws = [Value(v) for v in w]
    loss = Value(0.0)
    for i in range(len(y)):
        pred = sum((ws[j] * float(X[i, j]) for j in range(len(w))), Value(0.0))
        loss = loss + (pred - float(y[i])) ** 2
    loss = loss * (1.0 / len(y))
    order = loss.backward()
    return np.array([v.grad for v in ws]), len(order)


def linreg_data(n=200, d=5, seed=44):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, d))
    y = X @ np.array([1.0, -2.0, 0.5, 3.0, 0.0]) + 0.1 * rng.normal(size=n)
    w = rng.normal(size=d)
    return X, y, w


X, y, w = linreg_data()
gv, nodes = linreg_grad_values(X, y, w)
gn = linreg_grad_numpy(X, y, w)
print(nodes, 'nodes; max difference', np.abs(gv - gn).max())

4408 nodes; max difference 5.329070518200751e-15


## 10. LeetCode 150 - Evaluate Reverse Polish Notation, and LeetCode 566 - Reshape the Matrix

In [11]:
# ---------------------------------------------------------------- 6. LeetCode 150
class Solution150:
    """Evaluate Reverse Polish Notation; division truncates toward zero."""

    def evalRPN(self, tokens):
        st = []
        for t in tokens:
            if t in ('+', '-', '*', '/'):
                b, a = st.pop(), st.pop()
                if t == '+':
                    st.append(a + b)
                elif t == '-':
                    st.append(a - b)
                elif t == '*':
                    st.append(a * b)
                else:
                    st.append(int(a / b))      # truncate, NOT a // b (which floors)
            else:
                st.append(int(t))
        return st[0]


def eval_rpn_floor(tokens):
    st = []
    for t in tokens:
        if t in ('+', '-', '*', '/'):
            b, a = st.pop(), st.pop()
            st.append({'+': a + b, '-': a - b, '*': a * b, '/': a // b}[t])
        else:
            st.append(int(t))
    return st[0]


def rpn_graph(tokens, env):
    """The same stack machine over Value objects: evaluating builds the graph."""
    st = []
    for t in tokens:
        if t in ('+', '-', '*', '/'):
            b, a = st.pop(), st.pop()
            st.append({'+': lambda: a + b, '-': lambda: a - b,
                       '*': lambda: a * b, '/': lambda: a / b}[t]())
        else:
            st.append(env[t] if t in env else Value(float(t)))
    return st[0]


RPN_EXAMPLE = ['10', '6', '9', '3', '+', '-11', '*', '/', '*', '17', '+', '5', '+']


# ---------------------------------------------------------------- 7. LeetCode 566
class Solution566:
    """Reshape the Matrix: read row by row, write row by row."""

    def matrixReshape(self, mat, r, c):
        m, n = len(mat), len(mat[0])
        if m * n != r * c:
            return mat
        out = [[0] * c for _ in range(r)]
        for k in range(m * n):
            out[k // c][k % c] = mat[k // n][k % n]
        return out


print(Solution150().evalRPN(RPN_EXAMPLE), ' with //:', eval_rpn_floor(RPN_EXAMPLE))
x, yv = Value(2.0), Value(-3.0)
r = rpn_graph(['x', 'y', '*', 'x', '+', 'y', '/'], {'x': x, 'y': yv}); r.backward()
print('graph:', r.data, x.grad, yv.grad)
print(Solution566().matrixReshape([[1, 2, 3], [4, 5, 6]], 3, 2))
print(np.arange(1, 7).reshape(2, 3).reshape(3, 2, order='F').tolist(), '<- column-major order is a different answer')

22  with //: 12
graph: 1.3333333333333333 0.6666666666666667 -0.2222222222222222
[[1, 2], [3, 4], [5, 6]]
[[1, 5], [4, 3], [2, 6]] <- column-major order is a different answer


## Tests

In [12]:
def check() -> None:
    assert broadcast_shape((3,), (3, 1)) == (3, 3)
    assert broadcast_shape((5, 1, 4), (3, 1)) == (5, 3, 4)
    y = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    assert abs(mse((y + 0.1).reshape(-1, 1).ravel(), y) - 0.01) < 1e-12
    assert mse((y + 0.1).reshape(-1, 1), y) > 1
    assert ((np.array([200], np.uint8) + np.array([100], np.uint8)) // 2).tolist() == [22]
    m = np.arange(12).reshape(3, 4)
    assert np.shares_memory(m.T, m) and not np.shares_memory(m.T.reshape(12), m)
    pts = np.random.default_rng(7).random((40, 3))
    assert np.allclose(pairwise_broadcast(pts), pairwise_gram(pts), atol=1e-7)
    assert np.allclose(pairwise_broadcast(pts), np.array(pairwise_loop(pts.tolist())))
    res, order = micrograd_example()
    assert res['L'] == (-8.0, 1.0) and res['a'][1] == 6.0 and res['b'][1] == -4.0
    assert res['c'][1] == -2.0 and res['f'][1] == 4.0 and order[-1] == 'L'
    fx3 = lambda p, q, r: (p * q + r).tanh() * p + (q / r).exp() - p ** 2
    assert gradcheck(fx3, [0.5, -1.2, 2.0]) < 1e-7
    x = Value(3.0)
    (x * x).backward()
    xa = AssignValue(3.0)
    (xa * xa).backward()
    assert x.grad == 6.0 and xa.grad == 3.0
    x = Value(3.0)
    q = x * 2
    (q * q).backward()
    assert x.grad == 24.0
    x = Value(3.0)
    q = x * 2
    (q * q).backward_naive()
    assert x.grad == 48.0
    xs, ys = make_line_data()
    w1, b1, h1 = fit_line(xs, ys)
    w2, b2, h2 = fit_line(xs, ys, zero_grad=False)
    assert abs(w1 - 3) < 0.1 and abs(b1 + 0.5) < 0.1 and h2[-1] > 100 * h1[-1]
    X, y2, w = linreg_data()
    gv, _ = linreg_grad_values(X, y2, w)
    assert np.allclose(gv, linreg_grad_numpy(X, y2, w), atol=1e-10)
    x = Dual(2.0, 1.0)
    assert abs((x * x.sin() + 3 * x).der - (math.sin(2) + 2 * math.cos(2) + 3)) < 1e-12
    sol = Solution150()
    assert sol.evalRPN(RPN_EXAMPLE) == 22 and eval_rpn_floor(RPN_EXAMPLE) == 12
    assert sol.evalRPN(['2', '1', '+', '3', '*']) == 9 and sol.evalRPN(['4', '13', '5', '/', '+']) == 6
    s566 = Solution566()
    for mat, r, c in (([[1, 2], [3, 4]], 1, 4), ([[1, 2, 3], [4, 5, 6]], 3, 2), ([[1, 2], [3, 4]], 2, 4)):
        got = s566.matrixReshape(mat, r, c)
        arr = np.array(mat)
        assert got == (arr.reshape(r, c).tolist() if arr.size == r * c else mat)


check()
print('all assertions passed')

all assertions passed
